# Mode Report

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "exploratory"))

In [2]:
import gcsfs
import pandas as pd
import numpy as np

from _utils import *
from process_ntd import *
from download_ntd import *
from chart_ideas import *

In [3]:
df_capex = load_capex()
df_opex = load_service_and_opex()

In [4]:
df_capex_ca = subset_california(df_capex)
df_opex_ca = subset_california(df_opex)

In [5]:
df_capex_ca = add_mode_group(df_capex_ca)
df_opex_ca = add_mode_group(df_opex_ca)

In [6]:
df_capex_ca.columns

Index(['key', 'ntd_id', 'year', 'legacy_ntd_id', 'mode', 'mode_full_name',
       'agency_status', 'census_year', 'last_report_year', 'reporter_type',
       'reporting_module', 'uace_code', 'uza_area_sq_miles', 'uza_name',
       'uza_population', 'total_capital_expenditures',
       'rolling_stock_expenditures', 'facilities_expenditures',
       'other_expenditures', '_2024_mode_status', 'source_agency',
       'source_city', 'source_state', 'mode_group'],
      dtype='object')

In [7]:
df_opex_ca.columns

Index(['key', 'ntd_id', 'mode', 'year', 'type_of_service',
       'unlinked_passenger_trips', 'vehicle_revenue_hours',
       'vehicle_revenue_miles', 'vehicles_operated_in_maxiumum_service',
       'passenger_miles_traveled', 'directional_route_miles',
       'operating_expenses_vehicle_operations',
       'operating_expenses_vehicle_maintenance',
       'operating_expenses_nonvehicle_maintenance',
       'operating_expenses_general_administration', 'operating_expenses_total',
       'fare_revenue', 'opex_per_vrh', 'opex_per_vrm', 'opex_per_upt',
       'upt_per_vrh', 'upt_per_vrm', 'farebox_recovery_ratio', 'agency_status',
       'census_year', 'last_report_year', 'mode_status', 'reporter_type',
       'reporting_module', 'uace_code', 'uza_area_sq_miles',
       'primary_uza_name', 'uza_population', 'source_agency', 'source_city',
       'source_state', 'upt_prior_year', 'upt_change_1yr',
       'upt_pct_change_1yr', 'mode_full_name', 'type_of_service_full_name',
       'service_typ

In [8]:
cpi_annual = get_annual_average_cpi(2015, 2024)

base_cpi = cpi_annual.loc[cpi_annual["year"] == 2024, "cpi"].iloc[0]

In [9]:
df_capex_wd_real = add_real_values(
    df_capex_ca,
    value_cols={
        "total_capital_expenditures": "capex_total_real",
        "rolling_stock_expenditures": "rolling_stock_real",
        "facilities_expenditures": "facilities_real",
        "other_expenditures": "other_real",
    },
    cpi_annual=cpi_annual,
    base_cpi=base_cpi
)

df_opex_wd_real = add_real_values(
    df_opex_ca,
    value_cols={
        "operating_expenses_total": "opex_total_real",
        "operating_expenses_vehicle_operations": "opex_vehicle_operations_real",
        "operating_expenses_vehicle_maintenance": "opex_vehicle_maintenance_real",
        "operating_expenses_nonvehicle_maintenance": "opex_nonvehicle_maintenance_real",
        "operating_expenses_general_administration": "opex_general_administration_real",
    },
    cpi_annual=cpi_annual,
    base_cpi=base_cpi
)

## 1. Trend & composition

### Ridership, OPEX, CAPEX, VRH and VRM Trend

In [10]:
ridership_trend = mode_trend_chart(prep_for_mode_trend(df_opex_ca, "unlinked_passenger_trips"), "unlinked_passenger_trips").properties(title="Unlinked Passenger Trips")
opex_trend = mode_trend_chart(prep_for_mode_trend(df_opex_ca, "operating_expenses_total"), "operating_expenses_total").properties(title="Operating Expenditures (Total)")
capex_trend = mode_trend_chart(prep_for_mode_trend(df_capex_ca, "total_capital_expenditures"), "total_capital_expenditures").properties(title="Capital Expenditures (Total)")
vrh_trend = mode_trend_chart(prep_for_mode_trend(df_opex_ca, "vehicle_revenue_hours"), "vehicle_revenue_hours").properties(title="Vehicle Revenue Hours")
vrm_trend = mode_trend_chart(prep_for_mode_trend(df_opex_ca, "vehicle_revenue_miles"), "vehicle_revenue_miles").properties(title="Vehicle Revenue Miles")

mode_report_grid = (
    alt.vconcat(
        ridership_trend.properties(width=900),
        alt.hconcat(opex_trend, capex_trend),
        alt.hconcat(vrh_trend, vrm_trend)
    )
    .resolve_scale(color="shared")
    .configure_view(strokeWidth=0, fill="white")
    .configure_axis(labelFont="Arial", titleFont="Arial")
    .configure_title(font="Arial")
)

mode_report_grid


mode_report_grid


/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want indepe

alt.VConcatChart(...)

### Expenses Breakdown by Mode for Latest NTD Year

In [11]:
opex_chart = breakdown_chart(
    df_opex_ca,
    group_column = "mode_group", 
    year_column = "year",
    year=2024,
    component_columns=[
        "operating_expenses_vehicle_operations",
        "operating_expenses_vehicle_maintenance",
        "operating_expenses_nonvehicle_maintenance",
        "operating_expenses_general_administration",
    ],
    title="Operating Expense Composition by Mode, 2024",
)

capex_chart = breakdown_chart(
    df_capex_ca,
    group_column = "mode_group", 
    year_column = "year",
    year=2024,
    component_columns=[
        "rolling_stock_expenditures",
        "facilities_expenditures",
        "other_expenditures",
    ],
    title="Capital Expenditure Composition by Mode, 2024",
)


breakdown_by_mode = alt.hconcat(opex_chart, capex_chart)
breakdown_by_mode

alt.HConcatChart(...)

## 2. Efficiency/ Performance

### Mode Performance (HHI, Spending Per Capita and Volatility)

In [12]:
# HHI Index
opex_hhi = compute_hhi(df_opex_wd_real,
                       component_columns=[
                           "opex_vehicle_operations_real","opex_vehicle_maintenance_real",
                           "opex_nonvehicle_maintenance_real", "opex_general_administration_real"
                       ], 
                       group_column="mode_group",
                       year_column="year"
                      )



capex_hhi = compute_hhi(df_capex_wd_real,
                        component_columns=["rolling_stock_real", "facilities_real", "other_real"],
                        group_column="mode_group",
                        year_column="year"
                       )


# Capital and Operating Spending Per Capita
opex_pc = spending_per_capita(df_opex_wd_real, "opex_total_real", "uza_population", "mode_group", "year")
capex_pc = spending_per_capita(df_capex_wd_real, "capex_total_real", "uza_population", "mode_group", "year")


# Volatility (year-over-year swings, per mode)
opex_vol = spending_volatility(df_opex_wd_real, "opex_total_real", "mode_group", "year", 2024)
capex_vol = spending_volatility(df_capex_wd_real, "capex_total_real", "mode_group", "year", 2024)

In [13]:
scorecard = (
    opex_hhi[opex_hhi["year"] == 2024]
    .merge(
        opex_pc[opex_pc["year"] == 2024],
        on=["year", "mode_group"]
    )
    .merge(
        opex_vol,
        on="mode_group"
    )
)

opex_scorecard = plot_scorecard(
    scorecard,
    metrics=["hhi", "per_capita", "volatility"],
    group_column="mode_group",
    titles=["Expense Concentration", "Spending per Capita", "Spending Volatility (Compared to Previous Year)"],
    x_labels=["Operating Expenditure Composition HHI", "Spending per Capita (USD)", "Average Absolute YoY Change (%)"],
    formats=[".3f", "$,.0f", ".1%"],
    main_title="OPEX Mode Performance 2024"
)

opex_scorecard



alt.HConcatChart(...)

<!-- ## Farebox Recovery by Transit Mode -->

In [14]:
capex_scorecard = (
    capex_hhi[capex_hhi["year"] == 2024]
    .merge(
        capex_pc[capex_pc["year"] == 2024],
        on=["year", "mode_group"]
    )
    .merge(
        capex_vol,
        on="mode_group"
    )
)

capex_scorecard = plot_scorecard(
    capex_scorecard,
    metrics=["hhi", "per_capita", "volatility"],
    group_column="mode_group",
    titles=["Expense Concentration", "Spending per Capita", "Spending Volatility (Compared to Previous Year)"],
    x_labels=["Capital Expenditure Composition HHI", "Spending per Capita (USD)", "Average Absolute YoY Change (%)"],
    formats=[".3f", "$,.0f", ".1%"],
    main_title="CAPEX Mode Performance 2024"
)

capex_scorecard


alt.HConcatChart(...)

### Cost-Efficiency by Mode

In [15]:
cost_efficiency_matrix = efficiency_matrix(
    df_opex_ca[df_opex_ca["year"] == 2024],
    metrics=["opex_per_vrh", "opex_per_vrm", "opex_per_upt"],
    group_column="mode_full_name",
    titles=["Cost per VRH", "Cost per VRM", "Cost per Trip"],
    normalize="zscore",
    reverse_scale=[True, True, True],
    chart_title="Cost-Efficiency by Mode, 2024",
    width=600,
    height=400
)

cost_efficiency_matrix

alt.Chart(...)

## 3. Cost-effectiveness

### Farebox Recovery Ratio by Mode

In [16]:
farebox_recovery_ridge = ridge_plot(df_opex_ca, "farebox_recovery_ratio", "mode_full_name",
    year_column="year", year=2024,
    step=30,
    overlap=1.5,
    chart_title="Farebox Recovery Ratio Distribution by Transit Mode",
    chart_subtitle="2024 | One observation per California agency per mode")

farebox_recovery_ridge

alt.FacetChart(...)

### Cost Effectiveness Matrix

In [16]:
df_opex_ca["fare_revenue_per_trip"] = (
    df_opex_ca["fare_revenue"] / df_opex_ca["unlinked_passenger_trips"]
)

df_opex_ca["subsidy_per_trip"] = (
    df_opex_ca["operating_expenses_total"] - df_opex_ca["fare_revenue"]
) / df_opex_ca["unlinked_passenger_trips"]


cost_effectiveness_matrix = efficiency_matrix(
    df_opex_ca[df_opex_ca["year"] == 2024],
    metrics=["farebox_recovery_ratio", "fare_revenue_per_trip", "subsidy_per_trip"],
    group_column="mode_full_name",
    titles=["Farebox Recovery", "Fare Revenue per Trip", "Net Subsidy per Trip"],
    normalize="zscore",
    reverse_scale=[False, False, True],
    chart_title="Cost-Effectiveness by Mode, 2024",
    width=600,
    height=400
)

cost_effectiveness_matrix


alt.Chart(...)

### Operating Costs vs. Fare Revenue, Indexed to 2019

In [18]:
scissors_chart = indexed_scissors(
    df_opex_ca, "mode_group", "year",
    ["operating_expenses_total", "fare_revenue"], 2019,
    {"operating_expenses_total": "Operating Expenses", "fare_revenue": "Fare Revenue"},
    ["#2563EB", "#F59E0B"],
    "Operating Costs vs. Fare Revenue",
    "Both indexed to 2019 = 100; widening separation indicates divergent growth"
)

scissors_chart


alt.FacetChart(...)

## 4. Service Effectiveness

### Operating Efficiency: Cost vs. Service Utilization and Service Intensity

In [24]:
df_eff = df_opex_ca.copy()

# VRM
df_eff["opex_per_vrm"] = df_eff["operating_expenses_total"] / df_eff["vehicle_revenue_miles"]
df_eff["trips_per_vrm"] = df_eff["unlinked_passenger_trips"] / df_eff["vehicle_revenue_miles"]
df_eff["log_opex_vrm"] = np.log10(df_eff["opex_per_vrm"])
df_eff["log_trips_vrm"] = np.log10(df_eff["trips_per_vrm"])

opex_vrm = grouped_scatter(
    df_eff, "log_opex_vrm", "log_trips_vrm",
    group_column="mode_group", size_column="unlinked_passenger_trips",
    label_column="source_agency",
    tooltip_columns=["source_agency", "mode_group", "log_opex_vrm", "log_trips_vrm"],
    x_title="Log Operating Expense per Vehicle Revenue Mile",
    y_title="Log Passenger Trips per Vehicle Revenue Mile",
    chart_title="Operating Cost vs. Service Utilization — VRM-based"
)

# VRH
df_eff["opex_per_vrh"] = df_eff["operating_expenses_total"] / df_eff["vehicle_revenue_hours"]
df_eff["trips_per_vrh"] = df_eff["unlinked_passenger_trips"] / df_eff["vehicle_revenue_hours"]
df_eff["log_opex_vrh"] = np.log10(df_eff["opex_per_vrh"])
df_eff["log_trips_vrh"] = np.log10(df_eff["trips_per_vrh"])

opex_vrh = grouped_scatter(
    df_eff, "log_opex_vrh", "log_trips_vrh",
    group_column="mode_group", size_column="unlinked_passenger_trips",
    label_column="source_agency",
    tooltip_columns=["source_agency", "mode_group", "log_opex_vrh", "log_trips_vrh"],
    x_title="Log Operating Expense per Vehicle Revenue Hour",
    y_title="Log Passenger Trips per Vehicle Revenue Hour",
    chart_title="Operating Cost vs. Service Intensity — VRH-based"
)

service_utilization_chart = (
    alt.hconcat(opex_vrm, opex_vrh)
    .configure_view(strokeWidth=0)
    .configure_axis(labelFont="Arial", titleFont="Arial")
    .configure_title(font="Arial")
)

service_utilization_chart


/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/pandas/core/arrays/masked.py:691: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs2, **kwargs)
/home/jovyan/ntd-snapshot/.venv/lib/python3.11/site-packages/pandas/core/arrays/masked.py:691: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs2, **kwargs)


alt.HConcatChart(...)

### Fleet & Service Delivery

In [20]:
fleet_chart = (alt.Chart(df_opex_ca)
    .transform_calculate(fleet_utilization="datum.vehicle_revenue_hours / datum.vehicles_operated_in_maxiumum_service")
    .mark_bar()
    .encode(
        x=alt.X("fleet_utilization:Q", title="Vehicle revenue hours per vehicle"),
        y=alt.Y("mode_full_name:N", sort="-x", title=None),
        color=alt.Color("mode:N", legend=None),
        tooltip=["mode_full_name:N", alt.Tooltip("fleet_utilization:Q", title="Revenue hours / vehicle", format=".1f")]
    ).properties(width=650, height=400, title="How hard is the fleet working?"))

speed_chart = (alt.Chart(df_opex_ca)
    .transform_calculate(avg_speed="datum.vehicle_revenue_miles / datum.vehicle_revenue_hours")
    .mark_bar()
    .encode(
        x=alt.X("avg_speed:Q", title="Average operating speed (mph)"),
        y=alt.Y("mode_full_name:N", sort="-x", title=None),
        color=alt.Color("mode_full_name:N", legend=None),
        tooltip=["mode_full_name:N", alt.Tooltip("avg_speed:Q", title="Average speed", format=".1f")]
    ).properties(width=650, height=400, title="How fast does service actually run?"))

fleet_service_chart = (alt.hconcat(fleet_chart, speed_chart)
    .configure_view(strokeWidth=0)
    .configure_axis(labelFont="Arial", titleFont="Arial")
    .configure_title(font="Arial"))

display(fleet_service_chart)


alt.HConcatChart(...)